In [11]:
# Install dependencies

%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [12]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
# Create an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [14]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [15]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [16]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [17]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [18]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [19]:
import json

with open("010/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.333333333333333


In [20]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Object Key Prefix Filter\n\nHere's a Python function that filters S3 object keys by prefix:\n\n```python\ndef filter_s3_keys_by_prefix(keys: list, prefix: str) -> list:\n    \"\"\"\n    Filter AWS S3 object keys by a given prefix.\n    \n    Args:\n        keys (list): List of S3 object keys to filter\n        prefix (str): The prefix string to match against\n        \n    Returns:\n        list: List of S3 object keys that start with the given prefix\n        \n    Examples:\n        >>> keys = ['logs/2024/01/app.log', 'logs/2024/02/app.log', 'data/users.csv']\n        >>> filter_s3_keys_by_prefix(keys, 'logs/2024/01/')\n        ['logs/2024/01/app.log']\n        \n        >>> filter_s3_keys_by_prefix(keys, 'logs/')\n        ['logs/2024/01/app.log', 'logs/2024/02/app.log']\n    \"\"\"\n    return [key for key in keys if key.startswith(prefix)]\n\n\n# Example usage and test cases\nif __name__ == \"__main__\":\n    # Sample S3 object keys\n    s3_keys = [\n 